# 01 — Checkout Intent Dataset: EDA

Explore the Criteo Uplift v2.1 dataset (or synthetic fallback) and
establish the core statistical structure: treatment/control balance,
conversion rates, and — critically — whether **propensity** and **uplift**
identify the same users.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parents[2] / "src"))   # commerce-ml-lab/src
sys.path.insert(0, str(Path().resolve().parents[0] / "src"))   # 03_checkout_intent/src

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from commerce_ml.viz.plotting import apply_style, PALETTE
from commerce_ml.data.loaders import load_criteo, generate_criteo_like, CRITEO_DIR
from intent.features import FEATURE_COLS, temporal_split

apply_style()
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)

# Load real data if available, otherwise use synthetic fallback
try:
    df = load_criteo(sample_frac=0.10)
    DATA_SOURCE = "Criteo Uplift v2.1 (10% sample)"
    df["segment"] = "unknown"   # real data has no ground-truth segment
except FileNotFoundError:
    df = generate_criteo_like(n_rows=200_000, random_state=42)
    DATA_SOURCE = "Synthetic Criteo-like (200k rows)"
    print("NOTE: Real Criteo data not found. Using synthetic fallback.")
    print("Run `make data-criteo` to download the real dataset.")

print(f"Data source: {DATA_SOURCE}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nTreatment rate:  {df['treatment'].mean():.1%}")
print(f"Conversion rate: {df['conversion'].mean():.2%}")
print(f"Visit rate:      {df['visit'].mean():.1%}")

## Treatment/control balance and conversion rates

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# Treatment balance
tc = df["treatment"].value_counts()
axes[0].bar(["Control (W=0)", "Treated (W=1)"], tc.values, color=[PALETTE[0], PALETTE[2]], alpha=0.85, edgecolor="white")
axes[0].set_ylabel("Count"); axes[0].set_title("Treatment assignment (RCT)")
for bar, v in zip(axes[0].patches, tc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500, f"{v:,}", ha="center", fontsize=8)

# Conversion rates by arm
for arm, label, color in [(0,"Control",PALETTE[0]),(1,"Treated",PALETTE[2])]:
    sub = df[df["treatment"]==arm]["conversion"]
    axes[1].bar(label, sub.mean(), color=color, alpha=0.85, edgecolor="white")
    axes[1].text(0 if arm==0 else 1, sub.mean()+0.001, f"{sub.mean():.2%}", ha="center", fontsize=9)
axes[1].set_ylabel("Conversion rate"); axes[1].set_title("Conversion rate by arm")
ate = df[df["treatment"]==1]["conversion"].mean() - df[df["treatment"]==0]["conversion"].mean()
axes[1].set_title(f"Conversion by arm  (ATE = {ate:+.3%})")

# Segment conversion rates (synthetic only)
if "segment" in df.columns and df["segment"].iloc[0] != "unknown":
    seg_rr = df.groupby("segment")["conversion"].mean().sort_values(ascending=False)
    colors = [PALETTE[2 if s=="persuadable" else 0 if s=="sure_thing" else 1 if s=="lost_cause" else 3] for s in seg_rr.index]
    axes[2].bar(seg_rr.index, seg_rr.values, color=colors, alpha=0.85, edgecolor="white")
    axes[2].set_ylabel("Conversion rate"); axes[2].set_title("Conversion rate by segment (synthetic)")
    for bar, v in zip(axes[2].patches, seg_rr.values):
        axes[2].text(bar.get_x()+bar.get_width()/2, v+0.002, f"{v:.1%}", ha="center", fontsize=8)
else:
    axes[2].hist(df["conversion"], bins=2, color=PALETTE[0], alpha=0.85)
    axes[2].set_title("Conversion distribution")

plt.tight_layout(); plt.savefig(RESULTS / "eda_overview.png", dpi=120); plt.show()

## The core insight: propensity ≠ uplift

The key point of this project: users most likely to convert (high propensity)
are NOT the users most responsive to an intervention (high uplift).

- **Sure-things** have the highest conversion rate — but they'd convert regardless of treatment.
- **Persuadables** have a lower baseline conversion rate — but treatment makes a real difference.

Targeting by propensity wastes budget on sure-things. Targeting by uplift
captures the persuadable segment that actually responds.

In [ ]:
if df["segment"].iloc[0] != "unknown":
    ctrl = df[df["treatment"]==0].groupby("segment")["conversion"].mean()
    trt  = df[df["treatment"]==1].groupby("segment")["conversion"].mean()
    cate = (trt - ctrl).rename("true_cate")
    prop = df.groupby("segment")["conversion"].mean().rename("propensity")
    seg_df = pd.concat([prop, cate], axis=1).sort_values("propensity", ascending=False)

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(seg_df))
    w_bar = 0.35
    ax.bar(x - w_bar/2, seg_df["propensity"],  w_bar, label="Propensity P(Y=1)", color=PALETTE[0], alpha=0.85)
    ax.bar(x + w_bar/2, seg_df["true_cate"],   w_bar, label="True CATE (uplift)", color=PALETTE[2], alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(seg_df.index)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_ylabel("Rate"); ax.legend()
    ax.set_title("Propensity vs. CATE by segment — sure-things have high propensity but near-zero uplift")
    plt.tight_layout(); plt.savefig(RESULTS / "eda_propensity_vs_cate.png", dpi=120); plt.show()

    print("\nKey numbers:")
    print(seg_df.round(4).to_string())
    print("\nConclusion: targeting by propensity would prioritise sure-things (propensity=high, cate≈0).")
    print("           Targeting by uplift would prioritise persuadables (propensity=medium, cate=high).")

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 6, figsize=(16, 5))
for i, col in enumerate(FEATURE_COLS):
    ax = axes[i // 6][i % 6]
    for arm, color in [(0, PALETTE[0]), (1, PALETTE[2])]:
        sub = df[df["treatment"]==arm][col]
        ax.hist(sub, bins=30, alpha=0.5, color=color, density=True)
    ax.set_title(col, fontsize=8); ax.tick_params(labelsize=6)
plt.suptitle("Feature distributions: Control (blue) vs Treated (red) — should be balanced in RCT", fontsize=9)
plt.tight_layout(); plt.savefig(RESULTS / "eda_feature_distributions.png", dpi=120); plt.show()

df.to_parquet(RESULTS / "uplift_data.parquet", index=False)
print(f"Saved dataset ({len(df):,} rows) to results/uplift_data.parquet")